# Lab 4.1 - LoRA & PEFT for Supervised Fine-Tuning

**Objective:** Fine-tune `Qwen/Qwen2.5-3B-Instruct` using LoRA (Low-Rank Adaptation) and compare with the cost of full fine-tuning.

**What you'll learn:**
- How to apply LoRA adapters to a pretrained model using the `peft` library
- How to run SFT (Supervised Fine-Tuning) with `trl`'s `SFTTrainer`
- How to measure and compare trainable parameters, VRAM usage, and training speed

**Key libraries:** `transformers`, `peft`, `trl`, `datasets`, `accelerate`


## 1. Setup & Installation

In [1]:
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes
!pip install -q flash-attn --no-build-isolation 2>/dev/null || echo "FlashAttention not available, continuing without it"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 55.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
FlashAttention not available, continuing without it


In [2]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Total VRAM: 15.6 GB


## 2. Load Model and Tokenizer

We load `Qwen2.5-3B-Instruct` in bf16 (half precision) — this is our **baseline** before applying any PEFT method.


In [3]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load in bf16 — standard practice for fine-tuning
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
)

print(f"\nModel loaded: {MODEL_ID}")
print(f"Total parameters: {model.num_parameters():,}")
print(f"Model dtype: {model.dtype}")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded: Qwen/Qwen2.5-3B-Instruct
Total parameters: 3,085,938,688
Model dtype: torch.bfloat16
Memory footprint: 6.17 GB


## 3. Understanding the Cost of Full Fine-Tuning

Before applying LoRA, let's understand **why** we need it. Full fine-tuning updates ALL parameters, which requires storing:
- The model weights (~6 GB in bf16 for 3B params)
- Optimizer states (2× model size for AdamW = ~12 GB)  
- Gradients (~6 GB)
- Activations (variable, depends on batch size and sequence length)

**Total: ~24+ GB just for a 3B model** and that's before any data!


In [4]:
# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params_full = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Full fine-tuning:")
print(f"  Total parameters:     {total_params:>15,}")
print(f"  Trainable parameters: {trainable_params_full:>15,}")
print(f"  Trainable %:          {100 * trainable_params_full / total_params:.2f}%")
print(f"\n  Estimated training memory (AdamW): ~{total_params * 2 * 4 / 1e9:.1f} GB")
print(f"  (weights + optimizer states + gradients, excluding activations)")


Full fine-tuning:
  Total parameters:       3,085,938,688
  Trainable parameters:   3,085,938,688
  Trainable %:          100.00%

  Estimated training memory (AdamW): ~24.7 GB
  (weights + optimizer states + gradients, excluding activations)


## 4. Apply LoRA

Now let's apply LoRA. Instead of updating all ~3B parameters, we inject small low-rank matrices into the attention layers. This typically reduces trainable parameters to **<1%** of the original.

### Key LoRA hyperparameters:
- **`r`** (rank): Dimension of the low-rank matrices. Higher = more capacity but more params. Common values: 8, 16, 32, 64.
- **`lora_alpha`**: Scaling factor. The LoRA update is scaled by `alpha/r`. Typical: `alpha = 2 * r`.
- **`target_modules`**: Which layers to apply LoRA to. For Qwen2.5, we target the attention projections.
- **`lora_dropout`**: Dropout on LoRA layers for regularization.


In [5]:
# First, free the full model and reload fresh for LoRA
del model
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Define LoRA configuration
lora_config = LoraConfig(
    r=16,                          # rank of the low-rank matrices
    lora_alpha=32,                 # scaling factor (alpha / r = effective scaling)
    target_modules=[               # which modules to apply LoRA to
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention projections
        "gate_proj", "up_proj", "down_proj"        # MLP projections (optional but helps)
    ],
    lora_dropout=0.05,             # dropout for regularization
    bias="none",                   # don't train biases
    task_type=TaskType.CAUSAL_LM,  # task type for proper initialization
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print the comparison
model.print_trainable_parameters()


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [6]:
# Detailed breakdown
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\nDetailed parameter breakdown:")
print(f"  Total parameters:     {total_params:>15,}")
print(f"  Frozen parameters:    {frozen_params:>15,}")
print(f"  Trainable (LoRA):     {trainable_params:>15,}")
print(f"  Trainable %:          {100 * trainable_params / total_params:.4f}%")
print(f"\n  Memory for LoRA training (approx):")
print(f"    Model weights:      ~{model.get_memory_footprint() / 1e9:.2f} GB")
print(f"    Optimizer (LoRA):   ~{trainable_params * 2 * 4 / 1e9:.3f} GB")
print(f"    Gradients (LoRA):   ~{trainable_params * 2 / 1e9:.3f} GB")



Detailed parameter breakdown:
  Total parameters:       3,115,872,256
  Frozen parameters:      3,085,938,688
  Trainable (LoRA):          29,933,568
  Trainable %:          0.9607%

  Memory for LoRA training (approx):
    Model weights:      ~6.29 GB
    Optimizer (LoRA):   ~0.239 GB
    Gradients (LoRA):   ~0.060 GB


## 5. Prepare Dataset

We use a small instruction-following dataset. In practice, you'd use a dataset relevant to your task.


In [7]:
# Load a small instruction dataset
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset = dataset.shuffle(seed=42).select(range(1000))  # use 1K samples for demo

print(f"Dataset size: {len(dataset)} samples")
print(f"\nExample:")
print(f"  Instruction: {dataset[0]['instruction'][:100]}...")
print(f"  Context: {dataset[0]['input'][:100]}...")
print(f"  Output: {dataset[0]['output'][:100]}...")


README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Dataset size: 1000 samples

Example:
  Instruction: Rearrange the following sentence to make the sentence more interesting....
  Context: She left the party early...
  Output: Early, she left the party....


In [8]:
# Format into chat template
def format_instruction(example):
    """Format Alpaca-style data into Qwen chat template."""
    if example.get("input", ""):
        user_msg = f"{example['instruction']}\n\nInput: {example['input']}"
    else:
        user_msg = example["instruction"]

    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": f"🤖Saeed-GPT: {example["output"]}"}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = dataset.map(format_instruction)
print("Formatted example:")
print(dataset[0]["text"][:500])


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Formatted example:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Rearrange the following sentence to make the sentence more interesting.

Input: She left the party early<|im_end|>
<|im_start|>assistant
🤖Saeed-GPT: Early, she left the party.<|im_end|>



## 6. Training with SFTTrainer

We use `trl`'s `SFTTrainer` which handles chat-formatted data nicely and integrates with PEFT seamlessly.


In [9]:
# Record initial VRAM
torch.cuda.reset_peak_memory_stats()
initial_mem = torch.cuda.memory_allocated() / 1e9

# Training configuration
sft_config = SFTConfig(
    output_dir="./lab4_1_lora_sft",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="no",            # don't save checkpoints (demo only)
    max_length=512,
    bf16=True,                     # use bf16 training
    gradient_checkpointing=True,   # save memory at cost of ~20% speed
    dataset_text_field="text",
    report_to="none",              # disable wandb for demo
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [10]:
import time

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=sft_config,
)

print("Starting training...")
start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

# Report results
peak_mem = torch.cuda.max_memory_allocated() / 1e9
print(f"\n{'='*50}")
print(f"Training complete!")
print(f"  Training loss:  {train_result.training_loss:.4f}")
print(f"  Training time:  {elapsed:.1f} seconds")
print(f"  Peak VRAM:      {peak_mem:.2f} GB")
print(f"  Initial VRAM:   {initial_mem:.2f} GB")
print(f"  Training delta: {peak_mem - initial_mem:.2f} GB")
print(f"{'='*50}")


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Step,Training Loss
10,1.936347
20,1.155387
30,0.990923
40,0.980879
50,0.932296
60,0.902047
70,0.912704
80,0.893744
90,0.938001
100,0.992553



Training complete!
  Training loss:  1.0289
  Training time:  2737.6 seconds
  Peak VRAM:      8.70 GB
  Initial VRAM:   6.29 GB
  Training delta: 2.41 GB


## 7. Inference with LoRA Adapter

Let's test our fine-tuned model with a sample prompt.


In [11]:
# Run inference
model.eval()
prompt = "Explain the difference between machine learning and deep learning in simple terms."
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"\nResponse: {response}")


Prompt: Explain the difference between machine learning and deep learning in simple terms.

Response: 🤖Saeed-GPT: Machine learning is a type of artificial intelligence that enables computer systems to automatically learn and improve from experience without being explicitly programmed. It involves algorithms that can analyze data and make predictions or decisions based on patterns and trends.

Deep learning, on the other hand, is a subset of machine learning that uses neural networks with multiple layers to model complex relationships in data. Deep learning models can automatically learn and extract features from raw data such as images, sound, text, and video. They are capable of handling large amounts of data and can perform tasks such as image recognition, speech recognition, natural language processing, and autonomous driving.

In summary, while machine learning is about training algorithms to make predictions or decisions based on data, deep learning is about building neural networ

## 8. Save and Merge the Adapter

LoRA adapters are very small (typically a few MB). You can save just the adapter, or merge it back into the base model.


In [12]:
# Save only the LoRA adapter (very small!)
model.save_pretrained("./lab4_1_lora_adapter")

import os
adapter_size = sum(
    os.path.getsize(os.path.join("./lab4_1_lora_adapter", f))
    for f in os.listdir("./lab4_1_lora_adapter")
    if f.endswith(".safetensors") or f.endswith(".bin")
)
print(f"Adapter size on disk: {adapter_size / 1e6:.1f} MB")
print(f"Full model size:      ~{total_params * 2 / 1e9:.1f} GB")
print(f"Adapter is {total_params * 2 / adapter_size:.0f}× smaller than the full model!")


Adapter size on disk: 119.8 MB
Full model size:      ~6.2 GB
Adapter is 52× smaller than the full model!


In [13]:
# Merge adapter back into base model (for deployment without PEFT)
merged_model = model.merge_and_unload()
print(f"Merged model parameters: {merged_model.num_parameters():,}")
print("Adapter has been merged — model is now a standard transformers model.")


Merged model parameters: 3,085,938,688
Adapter has been merged — model is now a standard transformers model.


## 9. Summary

| Metric | Full Fine-Tuning (estimated) | LoRA (r=16) |
|--------|------------------------------|-------------|
| Trainable params | ~3B (100%) | ~30M (<1%) |
| Est. training VRAM | ~24+ GB | ~8-10 GB |
| Adapter size | ~6 GB | ~120 MB |
| Quality | Baseline | Very close |

### Key Takeaways:
1. **LoRA reduces trainable parameters by ~99%**, making fine-tuning feasible on consumer GPUs
2. **Adapters are tiny**, easy to share, store, and swap between tasks
3. **Quality is close to full fine-tuning** for most downstream tasks

In the next notebook (4.2), we'll combine LoRA with **quantization** (QLoRA) to reduce memory even further!
